# 16 — A2A Protocol: Agent-to-Agent Communication

The **A2A (Agent-to-Agent) Protocol** lets you expose any Strands agent as a network service and invoke it from other agents or clients. This enables multi-agent architectures where agents run as independent services and communicate over HTTP.

In this tutorial you will:
1. Wrap a Strands agent as an **A2A server**
2. Invoke it remotely with **A2AAgent**
3. Discover its capabilities via the **agent card**
4. Stream responses in real time
5. Compose remote agents as **tools** for an orchestrator

## Prerequisites

You need AWS credentials configured for Amazon Bedrock access. See [01-first-agent](../01-first-agent/) if you haven't set this up.

In [ ]:
%pip install -U 'strands-agents[a2a]' 'strands-agents-tools[a2a_client]' strands-agents-tools -q

## Section 1: Introduction — What is A2A?

When agents need to collaborate, they can run as separate services and talk to each other over the network. The A2A protocol provides:

- **A2AServer** — wraps any `Agent` and serves it over HTTP
- **Agent cards** — JSON metadata at `/.well-known/agent-card.json` describing capabilities
- **A2AAgent** — a client that discovers and invokes remote agents
- **A2AClientToolProvider** — wraps remote agents as tools for other agents

This means you can build a calculator agent, a research agent, and a writing agent as independent services, then have an orchestrator agent call whichever one it needs.

## Section 2: A2AServer — Wrapping an Agent as a Service

Let's create a simple calculator agent and serve it over A2A. The server automatically generates an agent card from the agent's `name` and `description`.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result.

    Args:
        expression: A mathematical expression to evaluate (e.g., '2 + 2', '10 ** 6').
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


model = BedrockModel(model_id="us.amazon.nova-lite-v1:0")

calculator_agent = Agent(
    model=model,
    tools=[calculator],
    name="Calculator Agent",
    description="An agent that evaluates mathematical expressions.",
    callback_handler=None,
)

print(f"Agent created: {calculator_agent.name}")
print(f"Tools: {[t.__name__ if hasattr(t, '__name__') else str(t) for t in calculator_agent.tool_names]}")

Now start the A2A server in a background thread so we can continue using this notebook:

In [ ]:
import threading
import time

from strands.multiagent.a2a import A2AServer

a2a_server = A2AServer(agent=calculator_agent, port=9000, enable_a2a_compliant_streaming=True)

server_thread = threading.Thread(target=a2a_server.serve, daemon=True)
server_thread.start()
time.sleep(2)  # wait for server to start

print("A2A server running at http://127.0.0.1:9000")
print("Agent card available at http://127.0.0.1:9000/.well-known/agent-card.json")

## Section 3: A2AAgent — Invoking a Remote Agent

`A2AAgent` is the client side. Point it at the server endpoint and call it just like a local agent — it returns an `AgentResult`.

In [ ]:
from strands.agent.a2a_agent import A2AAgent

a2a_client = A2AAgent(endpoint="http://127.0.0.1:9000")

result = a2a_client("What is 10 ** 6?")
print("Response:", result.message)

The remote agent received the message, used its `calculator` tool, and returned the result — all over HTTP. Let's try a more complex expression:

In [ ]:
result = a2a_client("Calculate the square root of 144 and then multiply it by 7")
print("Response:", result.message)

## Section 4: Agent Card Discovery

Every A2A server publishes an **agent card** — a JSON document describing the agent's name, description, capabilities, and skills. Skills are auto-derived from the agent's tools.

Let's fetch and inspect it:

In [ ]:
import json
import urllib.request

card_url = "http://127.0.0.1:9000/.well-known/agent-card.json"
with urllib.request.urlopen(card_url) as resp:
    agent_card = json.loads(resp.read())

print("=== Agent Card ===")
print(json.dumps(agent_card, indent=2))

Key fields in the agent card:
- **name** / **description** — from the `Agent` constructor
- **skills** — auto-generated from the agent's tools (each tool becomes a skill)
- **capabilities** — indicates streaming support and other features

This is how clients discover what a remote agent can do before invoking it.

## Section 5: Streaming Patterns

For long-running tasks, you can stream responses as they're generated. The `stream_async` method yields `A2AStreamEvent` objects:

In [ ]:
import asyncio


async def stream_example():
    print("Streaming response:")
    print("-" * 40)
    async for event in a2a_client.stream_async(
        "Explain step by step how to calculate 25 * 48"
    ):
        if "data" in event:
            print(event["data"], end="", flush=True)
    print("\n" + "-" * 40)
    print("Stream complete.")


await stream_example()

Streaming is enabled by default on the server. Each event contains a chunk of the response text, letting you display results progressively in a UI or CLI.

## Section 6: A2AClientToolProvider — Remote Agents as Tools

The most powerful pattern: wrap remote A2A agents as **tools** that another agent can call. `A2AClientToolProvider` discovers agents from their URLs and exposes them as callable tools.

This lets you build an orchestrator that delegates to specialized remote agents:

In [ ]:
from strands_tools.a2a_client import A2AClientToolProvider

# Discover remote agents and wrap them as tools
provider = A2AClientToolProvider(known_agent_urls=["http://127.0.0.1:9000"])

print(f"Discovered {len(provider.tools)} remote agent tool(s):")
for t in provider.tools:
    name = t.__name__ if hasattr(t, "__name__") else str(t)
    print(f"  - {name}")

In [ ]:
# Create an orchestrator agent that uses the remote calculator as a tool
orchestrator = Agent(
    model=model,
    tools=provider.tools,
    system_prompt="You are a helpful assistant. Use the available tools to answer questions.",
    callback_handler=None,
)

result = orchestrator("What is 123 * 456?")
print("Orchestrator response:", result.message)

The orchestrator doesn't know how to do math itself — it delegates to the remote Calculator Agent via A2A, gets the result, and presents it to the user.

## What You Learned

- **A2AServer** wraps any Strands agent and serves it over HTTP with an auto-generated agent card
- **A2AAgent** invokes remote agents and returns `AgentResult` just like local calls
- **Agent cards** at `/.well-known/agent-card.json` advertise capabilities and skills
- **Streaming** with `stream_async` delivers responses progressively
- **A2AClientToolProvider** wraps remote agents as tools, enabling orchestrator patterns

### Next Steps

- Deploy A2A servers as containers or Lambda functions (see [02-deploy](../../02-deploy/))
- Combine with [11-swarm](../11-swarm/) for dynamic multi-agent teams
- Add [05-guardrails](../05-guardrails/) to your A2A agents for content filtering